# Graha semantic segmentation inference
This notebook runs Graha/Lunar-FM semantic segmentation on Pipeline-generated Lunar WAC datacubes. Input selection follows the same `DATA_DICT` format as `semantic_ibm_train.ipynb`.

The raw datacube helper canonicalizes WAC bands to VIS (5) followed by UV (2), so native Graha `vis-uv` normalization can be used even though the source GeoTIFF stores UV first.

# Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
logging.getLogger('rasterio._env').setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from tqdm import tqdm

In [ ]:
# Run from lfm/notebooks, or adjust this for your HPC checkout.
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from lfm.all_models.sem_seg import build_graha_notebook_configs
from lfm.all_models.all_tasks.utils.common import _extract_logits
from lfm.all_models.all_tasks.data.normalization import load_terramind_pretraining_stats
from lfm.full_model.sem_seg import semantic_graha_components
from lfm.toy_model.sem_seg.data_cube_inference import (
    create_binary_colormap,
    get_datacube_data,
    sliding_window_inference,
)
print('Successfully imported LFM and Graha modules')

# User configuration
Use the same modality-oriented `DATA_DICT` structure as the training notebook. `selected_modalities` controls which canonical WAC/static groups are passed to inference, and `band_filters` uses indices local to each group.

For raw WAC datacubes, the canonical groups are `vis: [0..4]`, `uv: [0..1]`, and `static: [0..N-1]` after the helper's static-band filter. Native `vis-uv` normalization is used when static is not selected. A non-native static subset uses Graha's flexible fused `wac` input path.

In [ ]:
INPUT_ROOT_DIR = Path('/explore/nobackup/projects/lfm/model_inputs/inference/WAC_Processed_AOI')
GRAHA_PRETRAIN_DIR = Path('/explore/nobackup/projects/lfm/gabby/Lunar-FM/experiments/lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05')
GRAHA_LIGHTNING_CHECKPOINT = Path('/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/graha/model.ckpt')
OUTPUT_DIR = Path('./outputs/inference')

DATA_DICT = {
    'dataset_name': 'wac_static_inference',
    'data_dir': str(INPUT_ROOT_DIR),
    'dataset_modality': 'wac_static',
    'selected_modalities': ['vis', 'uv'],
    'band_filters': {
        'vis': [0, 1, 2, 3, 4],
        'uv': [0, 1],
        "static": [  # Keep all 63 static bands; remove individual indices here for ablation tests.
            0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
            10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
            20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
            30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
            40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
            50, 51, 52, 53, 54, 55, 56, 57, 58, 59,
            60, 61, 62,
        ],
    },
    'excluded_nodata_values': [
        -32768.0,
        -3.4028226550889045e38,
        -3.4028230607370965e38,
        -3.4028234663852886e38,
    ],
}

Non-configurable values, but values that need to be set early.

In [ ]:
MODEL_NATIVE_SIZE = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load and configure the input data

In [ ]:
# The helper returns WAC as VIS+UV, then filtered lola_kaguya static bands.
images_all, file_pairs = get_datacube_data(
    INPUT_ROOT_DIR, band_filter=None, verbose=True, verify_bands=True
)
if images_all.size == 0:
    raise ValueError('No valid WAC/Static datacube pairs were found.')

def selected_raw_band_indices(data_dict, static_count):
    layout = {'vis': list(range(5)), 'uv': list(range(5, 7)), 'static': list(range(7, 7 + static_count))}
    selected = data_dict['selected_modalities']
    filters = data_dict.get('band_filters', {})
    indices = []
    for modality in selected:
        available = layout[modality]
        local = filters.get(modality, list(range(len(available))))
        indices.extend(available[index] for index in local)
    return indices

static_count = max(int(images_all.shape[1]) - 7, 0)
BAND_FILTER = selected_raw_band_indices(DATA_DICT, static_count)
images_raw = images_all[:, BAND_FILTER]
n_channels = int(images_raw.shape[1])
selected = DATA_DICT['selected_modalities']
static_filter = DATA_DICT.get('band_filters', {}).get('static', list(range(static_count)))
static_selected_count = len(static_filter)
if selected == ['vis', 'uv']:
    GRAHA_BACKEND_MODALITIES = ['vis', 'uv']
    GRAHA_INPUT_MODE = 'vis-uv'
elif selected == ['vis', 'uv', 'static'] and static_selected_count == 63:
    GRAHA_BACKEND_MODALITIES = ['vis', 'uv', 'static']
    GRAHA_INPUT_MODE = 'vis-uv-static'
else:
    GRAHA_BACKEND_MODALITIES = ['wac']
    GRAHA_INPUT_MODE = 'single'
if GRAHA_BACKEND_MODALITIES == ['vis', 'uv'] or GRAHA_BACKEND_MODALITIES == ['vis'] or GRAHA_BACKEND_MODALITIES == ['uv']:
    NORMALIZATION_MODALITY = 'vis-uv'
elif GRAHA_BACKEND_MODALITIES == ['vis', 'uv', 'static']:
    NORMALIZATION_MODALITY = 'vis-uv-static'
else:
    NORMALIZATION_MODALITY = None
print(f'Input modalities: {selected}; channels: {n_channels}')
print(f'Graha backend: {GRAHA_BACKEND_MODALITIES}')

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
notebook_configs = build_graha_notebook_configs(
    output_dir=OUTPUT_DIR, data_root=INPUT_ROOT_DIR,
    graha_base_output_dir=OUTPUT_DIR, graha_pretrain_dir=GRAHA_PRETRAIN_DIR,
    graha_lightning_checkpoint=GRAHA_LIGHTNING_CHECKPOINT, data_dict=DATA_DICT,
    normalization_modality=NORMALIZATION_MODALITY,
    graha_input_modality_mode=GRAHA_INPUT_MODE,
    graha_backend_modalities=GRAHA_BACKEND_MODALITIES, validate_paths=False,
)
config = notebook_configs.experiment_config
graha_config = notebook_configs.graha_config
deps = notebook_configs.dependencies
if NORMALIZATION_MODALITY is not None:
    means, stds = load_terramind_pretraining_stats(
        graha_config.modality_info,
        normalization_modality=NORMALIZATION_MODALITY,
        band_filter=BAND_FILTER,
    )
else:
    means, stds = None, None

task_cls = semantic_graha_components.make_downstream_shape_segmentation_task_class(
    deps['LunarShapeSegmentationTask']
)
sample_batch = {'image': torch.zeros(1, n_channels, MODEL_NATIVE_SIZE, MODEL_NATIVE_SIZE)}
graha_task = semantic_graha_components.create_task(graha_config, task_cls, sample_batch).to(device)
semantic_graha_components.inspect_backbone(graha_task)
semantic_graha_components.load_lightning_checkpoint_state(graha_task, GRAHA_LIGHTNING_CHECKPOINT, 'Graha')
graha_task.eval()

class GrahaLogitModel(nn.Module):
    def __init__(self, task):
        super().__init__()
        self.task = task

    def forward(self, image):
        return _extract_logits(self.task(image))

model = GrahaLogitModel(graha_task).to(device).eval()
print('Successfully loaded Graha semantic checkpoint')

# Inference helpers

In [ ]:
def preprocess_datacubes(images_raw, means=None, stds=None):
    images_hwc = np.transpose(images_raw, (0, 2, 3, 1)).astype(np.float32)
    images_graha = np.empty_like(images_hwc)
    for index, image in enumerate(tqdm(images_hwc, desc='Preprocessing datacubes')):
        if means is not None and stds is not None:
            images_graha[index] = (image - np.asarray(means).reshape(1, 1, -1)) / np.asarray(stds).reshape(1, 1, -1)
        else:
            band_min = np.nanmin(image, axis=(0, 1), keepdims=True)
            band_max = np.nanmax(image, axis=(0, 1), keepdims=True)
            denominator = np.where(band_max > band_min, band_max - band_min, 1.0)
            images_graha[index] = 2.0 * ((image - band_min) / denominator) - 1.0
    return images_graha

def plot_inference_results(images_graha, preds, file_pairs, output_dir, n_channels):
    fig, axes = plt.subplots(2, len(file_pairs), figsize=(6 * len(file_pairs), 10), squeeze=False)
    for index, ((wac_file, _), image, pred) in enumerate(zip(file_pairs, images_graha, preds)):
        axes[0, index].imshow(image[:, :, 0], cmap='gray')
        axes[0, index].set_title(Path(wac_file).stem)
        axes[1, index].imshow(create_binary_colormap(pred))
        axes[1, index].set_title(f'Graha prediction ({int(pred.sum()):,} positive pixels)')
        for row in axes[:, index]:
            row.axis('off')
    fig.suptitle(f'Graha inference: {n_channels} input channels', y=1.0)
    fig.tight_layout()
    fig.savefig(output_dir / 'graha_inference_viz.png', dpi=150, bbox_inches='tight')
    plt.show()
    return fig

In [ ]:
images_graha = preprocess_datacubes(images_raw, means=means, stds=stds)
preds, probabilities = sliding_window_inference(images_graha, model=model, device=device, target_size=MODEL_NATIVE_SIZE, n_channels=n_channels)
fig = plot_inference_results(images_graha, preds, file_pairs, OUTPUT_DIR, n_channels)